In [2]:
import os
import sys
import pandas as pd
sys.path.append(os.path.abspath(os.path.join('..')))

train = pd.read_csv("../data/raw/train_BRCpofr.csv.zip", compression="zip")
test = pd.read_csv("../data/raw/test_koRSKBP.csv.zip", compression="zip")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import TransformedTargetRegressor
from xgboost import XGBRegressor

# Building a pipeline for the preprocessing
# 1. Define feature processing pipelines
numeric_features = ['vintage', 'claim_amount']
categorical_features = ['type_of_policy', 'marital_status', 'qualification']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ],
    remainder='passthrough'  # Keep other columns unchanged or wahala for dey
)

# 2. Bundle preprocessing and the XGBRegressor model into a feature pipeline
xgbregressor_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor())
])

# 3. Wrap everything to handle your Yeo-Johnson Target automatically!
wrapped_model = TransformedTargetRegressor(
    regressor=xgbregressor_pipeline,
    transformer=PowerTransformer(method='yeo-johnson')
)

In [ ]:
from sklearn.model_selection import train_test_split

y = train.drop(columns=['id', 'cltv']).set_index('id')
X = train['cltv']

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)